# AML Monitoring — Typology Walkthrough with JMLSG Regulatory Commentary

**FinCrime-ML framework · AML domain**

This notebook demonstrates the end-to-end AML monitoring pipeline: from synthetic transaction generation through typology detection, graph-based anomaly scoring, unsupervised isolation forest, SAR trigger scoring, and alert fatigue evaluation.

Regulatory references are embedded throughout. Key frameworks:

| Framework | Relevance |
|---|---|
| POCA 2002 s.330 | Failure to disclose — criminal offence for regulated firms |
| JMLSG Part I Ch.5 | Transaction monitoring indicators and typology guidance |
| FATF Recommendations R.10, R.16, R.20 | CDD, wire transfer transparency, STR obligation |
| MLR 2017 Reg 28 | Enhanced Due Diligence requirements |
| FCA SYSC 6.3 / 10A | Alert management, automated decision audit records |

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from fincrime_ml.core.data.synth_aml import SyntheticAMLGenerator
from fincrime_ml.aml.typologies import TypologyEngine
from fincrime_ml.aml.graph import TransactionGraphBuilder
from fincrime_ml.aml.models.isolation_forest import AMLIsolationForest
from fincrime_ml.aml.sar_scorer import SARScorer, SARScorerConfig
from fincrime_ml.aml.evaluation import AlertFatigueEvaluator

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 42
print("Imports OK")

---
## 1. Synthetic AML Data

**JMLSG Part I para 5.3.1** requires that transaction monitoring systems are calibrated against realistic transaction patterns. We use `SyntheticAMLGenerator` to produce a labelled dataset with:

- Mule account seeds (structuring + layering injection)
- Realistic amount distributions (GBP, UK digital payments context)
- Suspicious rate configurable to reflect typical AML prevalence (0.5–3%)

In [ ]:
gen = SyntheticAMLGenerator(n_accounts=500, seed=SEED)
df = gen.generate(n_transactions=5_000, suspicious_rate=0.04)

print(f"Transactions : {len(df):,}")
print(f"Suspicious   : {df['is_suspicious'].sum():,} ({df['is_suspicious'].mean()*100:.1f}%)")
print(f"Columns      : {list(df.columns)}")
df.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Amount distribution
for label, grp in df.groupby("is_suspicious"):
    axes[0].hist(grp["amount_gbp"].clip(upper=15_000), bins=50, alpha=0.6,
                 label="Suspicious" if label else "Legitimate")
axes[0].set_title("Transaction Amount Distribution (GBP)")
axes[0].set_xlabel("Amount (GBP)")
axes[0].legend()

# Hour of day
for label, grp in df.groupby("is_suspicious"):
    counts = grp["hour_of_day"].value_counts().sort_index()
    axes[1].plot(counts.index, counts.values / len(grp),
                 label="Suspicious" if label else "Legitimate")
axes[1].set_title("Transaction Volume by Hour of Day")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Fraction of transactions")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 2. Typology Detection

**JMLSG Part I Ch.5** identifies three primary AML typologies that transaction monitoring systems should detect:

| Typology | JMLSG Reference | Key Indicator |
|---|---|---|
| **Structuring** | para 5.3.11 | Amounts just below reporting thresholds (GBP 8,500–9,950) |
| **Layering** | para 5.3.17 | Multi-hop fund movements through mule chains |
| **Integration** | para 5.3.14 | Funds re-entering the legitimate economy |

`TypologyEngine` applies rule-based detection to flag transactions matching these patterns.

In [ ]:
engine = TypologyEngine()
df_typed = engine.annotate(df)

typology_counts = df_typed["typology"].value_counts()
print("Typology distribution:")
print(typology_counts.to_string())

In [ ]:
# Suspicious rate by typology — JMLSG para 5.3.1 calibration check
typology_stats = (
    df_typed.groupby("typology")["is_suspicious"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "suspicious_rate", "sum": "n_suspicious", "count": "n_total"})
    .sort_values("suspicious_rate", ascending=False)
)
typology_stats["suspicious_rate"] = typology_stats["suspicious_rate"].map("{:.1%}".format)
print(typology_stats.to_string())

In [ ]:
# Structuring deep-dive — POCA 2002 s.330 threshold avoidance band
structuring_txns = df_typed[
    (df_typed["structuring_flag"] == True) | (df_typed["typology"] == "structuring")
]
print(f"Structuring-flagged transactions: {len(structuring_txns)}")
print(f"Amount range: GBP {structuring_txns['amount_gbp'].min():.2f} – {structuring_txns['amount_gbp'].max():.2f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(structuring_txns["amount_gbp"], bins=40, color="#c0392b", alpha=0.7)
ax.axvline(8_500, color="k", linestyle="--", label="POCA s.330 lower bound (GBP 8,500)")
ax.axvline(9_950, color="k", linestyle=":", label="POCA s.330 upper bound (GBP 9,950)")
ax.set_title("Structuring: Amount Distribution within POCA 2002 s.330 Band")
ax.set_xlabel("Amount (GBP)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Transaction Network Graph

**FATF R.16** (wire transfer transparency) and **FATF R.10** (CDD) both require understanding the relationships between sending and receiving entities. `TransactionGraphBuilder` constructs a directed NetworkX graph of entity relationships, enabling:

- Mule chain path detection
- Centrality-based anomaly identification (high betweenness = likely layering node)
- Flow deviation scoring (nodes with anomalous in/out ratios)

In [ ]:
builder = TransactionGraphBuilder()
G = builder.build(df_typed)

print(f"Graph nodes (accounts)   : {G.number_of_nodes():,}")
print(f"Graph edges (transactions): {G.number_of_edges():,}")

# Node-level features
node_features = builder.node_features(G, df_typed)
print(f"\nNode feature columns: {list(node_features.columns)}")
print(f"Node feature shape  : {node_features.shape}")
node_features.describe().T[["mean", "std", "min", "max"]].round(4)

In [ ]:
# Top nodes by betweenness centrality — potential layering hubs
# (FATF R.16: nodes processing funds from multiple sources are structurally suspicious)
if "betweenness_centrality" in node_features.columns:
    top_nodes = node_features.nlargest(10, "betweenness_centrality")[
        [c for c in ["betweenness_centrality", "in_degree", "out_degree",
                     "total_volume_sent", "total_volume_received"] if c in node_features.columns]
    ]
    print("Top 10 nodes by betweenness centrality (FATF R.16 layering indicators):")
    print(top_nodes.to_string())

---
## 4. Unsupervised Baseline — Isolation Forest

**JMLSG Part I para 5.3.1** notes that transaction monitoring may be deployed without labelled data in early-stage implementations. `AMLIsolationForest` addresses the **no-label scenario**: it trains purely on transaction features, inversely maps sklearn's anomaly scores to a normalised risk score in [0, 1], and produces SHAP explanations without requiring ground-truth labels.

This is the recommended baseline before investing in supervised labelling.

In [ ]:
# Train on unlabelled data — core no-label scenario
df_unlabelled = df_typed.drop(columns=["is_suspicious"])

iso_model = AMLIsolationForest(n_estimators=100)
iso_model.train(df_unlabelled)

iso_scores = iso_model.predict(df_unlabelled)
print(f"Scored {len(iso_scores):,} transactions")
print(iso_scores[["risk_score", "risk_tier"]].describe().round(4))

In [ ]:
# Post-hoc evaluation — using labels we withheld from training
iso_eval = iso_model.evaluate(df_typed, label_col="is_suspicious")
print(f"Isolation Forest (unsupervised) — post-hoc evaluation:")
print(f"  AUC-PR  : {iso_eval['auc_pr']:.4f}")
print(f"  ROC-AUC : {iso_eval['roc_auc']:.4f}")

In [ ]:
# Risk tier distribution
tier_counts = iso_scores["risk_tier"].value_counts().reindex(["CRITICAL","HIGH","MEDIUM","LOW"]).fillna(0)

fig, ax = plt.subplots(figsize=(8, 4))
colours = {"CRITICAL": "#c0392b", "HIGH": "#e67e22", "MEDIUM": "#f1c40f", "LOW": "#27ae60"}
bars = ax.bar(tier_counts.index, tier_counts.values,
              color=[colours[t] for t in tier_counts.index])
ax.bar_label(bars, fmt="%d")
ax.set_title("Isolation Forest — Risk Tier Distribution (Unsupervised)")
ax.set_ylabel("Transaction count")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP explanations — top reason codes for highest-risk transactions
top_risk = iso_scores.nlargest(20, "risk_score")
top_risk_df = df_unlabelled[df_unlabelled["transaction_id"].isin(top_risk["transaction_id"])]

explanations = iso_model.explain(top_risk_df)
print("SHAP top reason codes for 20 highest-risk transactions:")
print(explanations[["transaction_id", "top_reason_1", "top_reason_2", "top_reason_3"]].to_string(index=False))

---
## 5. SAR Trigger Scoring

The final step before MLRO referral is SAR trigger evaluation. **POCA 2002 s.330** creates a criminal offence for regulated firms that fail to disclose known or suspected money laundering. `SARScorer` operationalises that obligation:

| Trigger | Statutory Basis |
|---|---|
| HIGH_RISK_SCORE | JMLSG Part I para 5.3.1; FCA FCG 3.2 |
| STRUCTURING_AMOUNT | POCA 2002 s.330; JMLSG para 5.3.11 |
| MULE_INVOLVEMENT | MLR 2017 Reg 28; JMLSG para 5.3.17 |
| RAPID_MOVEMENT | FATF R.10; JMLSG para 5.3.7 |
| CHAIN_LAYERING | FATF R.16; JMLSG para 5.3.17 |
| SUSPICIOUS_TYPOLOGY | FATF R.20; JMLSG Part I Ch.5 |

Alerts are prioritised 1 (CRITICAL) → 3 (MEDIUM). Priority 1 alerts carry an automatic SAR filing recommendation.

In [ ]:
# Merge risk scores back onto the annotated transaction frame
scored_df = df_typed.merge(
    iso_scores[["transaction_id", "risk_score", "risk_tier"]],
    on="transaction_id",
    how="left",
)

sar_scorer = SARScorer(SARScorerConfig(alert_score_threshold=0.30, sar_score_threshold=0.65))
alerts = sar_scorer.score(scored_df)

print(f"Total transactions scored : {len(scored_df):,}")
print(f"Alerts generated          : {len(alerts):,} ({len(alerts)/len(scored_df)*100:.1f}%)")
print(f"SAR filing recommended    : {alerts['sar_recommended'].sum():,}")
print(f"Priority 1 (CRITICAL)     : {(alerts['priority'] == 1).sum():,}")
print(f"Priority 2 (HIGH)         : {(alerts['priority'] == 2).sum():,}")
print(f"Priority 3 (MEDIUM)       : {(alerts['priority'] == 3).sum():,}")

In [ ]:
# MI summary report — suitable for compliance committee presentation
report = sar_scorer.summary_report(alerts)
print("SAR MI Report:")
for k, v in report.items():
    print(f"  {k:<25}: {v}")

In [ ]:
# Top 5 Priority 1 alerts — MLRO review queue
p1 = alerts[alerts["priority"] == 1].head(5)
if len(p1) > 0:
    for _, row in p1.iterrows():
        print(f"Alert: {row['alert_id']}")
        print(f"  Transaction : {row['transaction_id']}")
        print(f"  Amount      : GBP {row['amount_gbp']:,.2f}")
        print(f"  Risk Score  : {row['risk_score']:.4f} ({row['risk_tier']})")
        print(f"  Triggers    : {row['trigger_reasons']}")
        print(f"  SAR Rec.    : {'YES' if row['sar_recommended'] else 'NO'}")
        print()

---
## 6. Alert Fatigue Evaluation

**FCA SYSC 6.3** requires that automated transaction monitoring systems are regularly reviewed for effectiveness. Excessive false positive rates cause **alert fatigue** — a condition where analysts become desensitised and miss genuine SAR cases.

**JMLSG Part I para 5.3.1** states that monitoring systems should be tuned to achieve an appropriate balance between detection sensitivity and false positive burden. `AlertFatigueEvaluator` quantifies this trade-off:

- **FPR at 80–99% sensitivity**: How many legitimate transactions are incorrectly flagged at each recall level?
- **Fatigue index**: What fraction of analyst-reviewed alerts are false positives?
- **Optimal threshold**: The F1-maximising operating point subject to minimum sensitivity.

In [ ]:
evaluator = AlertFatigueEvaluator()

y_true = df_typed["is_suspicious"].tolist()
risk_scores = scored_df["risk_score"].tolist()

eval_report = evaluator.evaluate(y_true, risk_scores)

print(f"Dataset: {eval_report['n_positives']} suspicious / {eval_report['n_negatives']} legitimate")
print(f"Base rate       : {eval_report['base_rate']*100:.1f}%")
print(f"AUC-PR          : {eval_report['auc_pr']:.4f}")
print(f"ROC-AUC         : {eval_report['roc_auc']:.4f}")
print(f"Optimal threshold: {eval_report['optimal_threshold']['value']:.4f}")
print(f"  Precision      : {eval_report['optimal_threshold']['precision']:.3f}")
print(f"  Recall         : {eval_report['optimal_threshold']['recall']:.3f}")
print(f"  FPR            : {eval_report['optimal_threshold']['fpr']:.3f}")
print(f"  Fatigue Index  : {eval_report['optimal_threshold']['fatigue_index']:.3f}")

In [ ]:
# FPR vs Sensitivity — the core alert fatigue trade-off table
print(f"{'Sensitivity':>12} {'Threshold':>10} {'FPR':>8} {'Precision':>10} {'Alert Rate':>11} {'Fatigue Idx':>12}")
print("-" * 65)
for target, entry in sorted(eval_report["sensitivity_analysis"].items()):
    print(
        f"{target*100:>11.0f}%"
        f"  {entry['threshold']:>9.4f}"
        f"  {entry['fpr']*100:>6.1f}%"
        f"  {entry['precision']*100:>8.1f}%"
        f"  {entry['alert_rate']*100:>9.1f}%"
        f"  {entry['fatigue_index']*100:>10.1f}%"
    )

In [ ]:
# Alert volume profile — threshold sweep
profile = evaluator.alert_volume_profile(y_true, risk_scores)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Precision and Recall vs threshold
axes[0].plot(profile["threshold"], profile["precision"], label="Precision", color="#2980b9")
axes[0].plot(profile["threshold"], profile["recall"], label="Recall (Sensitivity)", color="#e74c3c")
axes[0].plot(profile["threshold"], profile["f1"], label="F1", color="#27ae60", linestyle="--")
opt = eval_report["optimal_threshold"]["value"]
axes[0].axvline(opt, color="k", linestyle=":", alpha=0.7, label=f"Optimal ({opt:.3f})")
axes[0].set_xlabel("Score Threshold")
axes[0].set_ylabel("Metric value")
axes[0].set_title("Precision / Recall / F1 vs Threshold")
axes[0].legend()

# Right: Alert fatigue index vs threshold
axes[1].plot(profile["threshold"], profile["fatigue_index"] * 100,
             color="#c0392b", linewidth=2)
axes[1].axvline(opt, color="k", linestyle=":", alpha=0.7, label=f"Optimal ({opt:.3f})")
axes[1].set_xlabel("Score Threshold")
axes[1].set_ylabel("Fatigue Index (% of alerts that are FP)")
axes[1].set_title("Alert Fatigue Index vs Threshold\n(FCA SYSC 6.3 — monitoring system review)")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity curve (ROC-space) — FPR vs Sensitivity
curve = evaluator.sensitivity_curve(y_true, risk_scores)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(curve["fpr"] * 100, curve["sensitivity"] * 100,
        color="#2980b9", linewidth=2, label=f"AML Model (ROC-AUC={eval_report['roc_auc']:.3f})")
ax.plot([0, 100], [0, 100], "k--", alpha=0.4, label="Random classifier")
ax.set_xlabel("False Positive Rate (% of legitimate transactions flagged)")
ax.set_ylabel("Sensitivity / Recall (% of suspicious transactions detected)")
ax.set_title("Sensitivity Curve — Alert Fatigue Trade-off\n(JMLSG Part I para 5.3.1 calibration view)")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Regulatory Summary

The following table summarises which regulatory obligations each pipeline component helps satisfy:

| Component | Regulatory Hook | Obligation Met |
|---|---|---|
| `SyntheticAMLGenerator` | JMLSG para 5.3.1 | Monitoring system calibration with realistic transaction patterns |
| `TypologyEngine` | JMLSG Ch.5, FATF R.20 | Detection of structuring, layering, and integration typologies |
| `TransactionGraphBuilder` | FATF R.10, R.16 | Entity relationship mapping; CDD and wire transfer transparency |
| `AMLIsolationForest` | MLR 2017 Reg 19 | Unsupervised baseline deployable without labelled training data |
| `SARScorer` | POCA 2002 s.330 | SAR filing recommendation with MLRO-ready audit output |
| `AlertFatigueEvaluator` | FCA SYSC 6.3 | Quantified false positive rate for monitoring system review |

All decision records produced by this pipeline are eligible for submission to FCA supervisory review under **FCA SYSC 10A** (automated decision systems audit trail requirements).

In [ ]:
# Audit log entries — FCA SYSC 10A compliance
print("Isolation Forest audit log:")
for entry in iso_model.audit_log:
    print(f"  [{entry['timestamp']}] {entry['event']} | supervised={entry.get('supervised')} | version={entry['version']}")

print("\nSAR scorer audit log:")
for entry in sar_scorer.audit_log:
    print(f"  [{entry['timestamp']}] {entry['event']} | n_alerts={entry.get('n_alerts_generated')} | n_sar={entry.get('n_sar_recommended')}")